In [ ]:
import sys, os

# Đường dẫn theo CLAUDE.md (chạy trên Kaggle):
#   UTILITY_PATH = "/kaggle/input/notebooks/laithetrung/waft-utility-script"
#   CODE_SRC     = "/kaggle/input/notebooks/laithetrung/waft-repo/JeDepth/train.sh"
#   DATASET_SRC  = "/kaggle/input/datasets/laithetrung/stereo-smallbaseline"
UTILITY_PATH = "/kaggle/input/notebooks/laithetrung/waft-utility-script"
REPO_PATH    = "/kaggle/input/notebooks/laithetrung/waft-repo/JeDepth"
DATASET_SRC  = "/kaggle/input/datasets/laithetrung/stereo-smallbaseline"

# Fallbacks cho convention `/kaggle/input/<slug>/...`
for p in ["/kaggle/input/waft-utility/kaggle/working", "/kaggle/input/waft-utility"]:
    if os.path.exists(p) and not os.path.exists(UTILITY_PATH):
        UTILITY_PATH = p
        break
for p in ["/kaggle/input/jedepth-code", "/kaggle/input/waft-stereo-repo"]:
    if os.path.exists(p) and not os.path.exists(REPO_PATH):
        REPO_PATH = p
        break
for p in ["/kaggle/input/stereo-smallbaseline"]:
    if os.path.exists(p) and not os.path.exists(DATASET_SRC):
        DATASET_SRC = p
        break

if os.path.exists(UTILITY_PATH):
    sys.path.insert(0, UTILITY_PATH)
if os.path.exists(REPO_PATH):
    sys.path.insert(0, REPO_PATH)
    os.chdir(REPO_PATH)

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("UTILITY_PATH:", UTILITY_PATH, os.path.exists(UTILITY_PATH))
print("REPO_PATH   :", REPO_PATH, os.path.exists(REPO_PATH))
print("DATASET_SRC :", DATASET_SRC, os.path.exists(DATASET_SRC))

In [ ]:
# ── Link the dataset to the path expected by the CSVs ──
DATASET_TARGET_ROOT = "/kaggle/working/data"
os.makedirs(DATASET_TARGET_ROOT, exist_ok=True)
target = os.path.join(DATASET_TARGET_ROOT, "stereo-smallbaseline")
if not os.path.exists(target):
    os.symlink(DATASET_SRC, target)
print("Dataset linked at", target)
print("Files:", os.listdir(target)[:5])

In [ ]:
# ── Launch training. Outputs (ckpts/, tb logs) lưu vào /kaggle/working ──
import subprocess, shutil
os.environ["PYTHONPATH"] = f"{REPO_PATH}:{UTILITY_PATH}:" + os.environ.get("PYTHONPATH", "")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

cmd = [
    "python", "main.py",
    "--config-file", "configs/Real/stereo_smallbaseline.yaml",
    "--num-gpus", "1",
    "--seed", "42",
    "DATASETS.ROOT", DATASET_TARGET_ROOT,
    "DATASETS.TRAIN_CSV", os.path.join(target, "train.csv"),
    "DATASETS.VAL_CSV",   os.path.join(target, "val.csv"),
    "TEST.TEST_IMAGES_DIR", os.path.join(REPO_PATH, "test_images"),
    "SOLVER.IMS_PER_BATCH", "4",
    "SOLVER.BASE_LR", "0.0002",
    "SOLVER.MAX_EPOCH", "60",
    "TEST.EVAL_EPOCH_PERIOD", "5",
    "DATASETS.CROP_SIZE", "[384,640]",
]
print("Launching:", " ".join(cmd))
subprocess.run(cmd, check=True)

# Copy ckpts ra /kaggle/working để persist
src_ckpt = os.path.join(REPO_PATH, "ckpts")
if os.path.exists(src_ckpt):
    shutil.copytree(src_ckpt, "/kaggle/working/ckpts", dirs_exist_ok=True)
print("Done. Checkpoints + TB logs ở /kaggle/working/ckpts")